In [1]:
import pandas as pd

In [ ]:
def load_fbref_csv(filepath):
    df = pd.read_csv(filepath, header = [0,1])
    df.columns = [
        '_'.join(col).strip()
        if 'Unnamed' not in col[0]
        else col[1].strip()
        for col in df.columns
    ]
    df = df.drop(index=0).reset_index(drop=True)
    return df

In [ ]:
df_standard = load_fbref_csv("../data/raw/standard.csv")

df_standard = df_standard.rename(columns={
    'Unnamed: 0_level_1': 'league',
    'Unnamed: 1_level_1': 'season',
    'Unnamed: 2_level_1': 'team',
    'Unnamed: 3_level_1': 'player',
    'nation_Unnamed: 4_level_1': 'nation',
    'pos_Unnamed: 5_level_1': 'pos',
    'age_Unnamed: 6_level_1': 'age',
    'born_Unnamed: 7_level_1': 'born',
    'Playing Time_MP': 'matches',
    'Playing Time_Starts': 'starts',
    'Playing Time_Min': 'minutes',
    'Playing Time_90s': '90s',
    'Performance_Gls': 'goals',
    'Performance_Ast': 'assists',
    'Performance_G+A': 'goal_contributions',
    'Performance_G-PK': 'goals_excl_pk',
    'Performance_PK': 'pk_scored',
    'Performance_PKatt': 'pk_attempted',
    'Performance_CrdY': 'yellow_cards',
    'Performance_CrdR': 'red_cards',
    'Per 90 Minutes_Gls': 'goals_p90',
    'Per 90 Minutes_Ast': 'assists_p90',
    'Per 90 Minutes_G+A': 'goal_contributions_p90',
    'Per 90 Minutes_G-PK': 'goals_excl_pk_p90',
    'Per 90 Minutes_G+A-PK': 'goal_contributions_excl_pk_p90'
})
print(df_standard.columns)
#print(df_standard.shape)
print(df_standard.head())

In [ ]:
df_shooting = load_fbref_csv("../data/raw/shooting.csv")

df_shooting = df_shooting.rename(columns={
    'Unnamed: 0_level_1': 'league',
    'Unnamed: 1_level_1': 'season',
    'Unnamed: 2_level_1': 'team',
    'Unnamed: 3_level_1': 'player',
    'nation_Unnamed: 4_level_1': 'nation',
    'pos_Unnamed: 5_level_1': 'pos',
    'age_Unnamed: 6_level_1': 'age',
    'born_Unnamed: 7_level_1': 'born',
    '90s_Unnamed: 8_level_1': '90s',
    'Standard_Gls': 'goals',
    'Standard_Sh': 'shots',
    'Standard_SoT': 'shots_on_target',
    'Standard_SoT%': 'shot_accuracy',
    'Standard_Sh/90': 'shots_p90',
    'Standard_SoT/90': 'shots_on_target_p90',
    'Standard_G/Sh': 'goals_per_shot',
    'Standard_G/SoT': 'goals_per_shot_on_target',
    'Standard_PK': 'pk_scored',
    'Standard_PKatt': 'pk_attempted'
})

print(df_shooting.columns)
#print(df_shooting.shape)
print(df_shooting.head())

In [ ]:
# Merging the data

shooting_cols = ['league', 'season', 'team', 'player', 
                 'shots', 'shots_on_target', 'shot_accuracy',
                 'shots_p90', 'shots_on_target_p90', 
                 'goals_per_shot', 'goals_per_shot_on_target']

df_merged = pd.merge(
    df_standard,
    df_shooting[shooting_cols],
    on=['league', 'season', 'team', 'player'],
    how='left'
)

print(df_merged.shape)
print(df_merged.columns)
print(df_merged.dtypes)

In [ ]:
# Processing the merged dataset

# Now let us remove players with playtime below 900 minutes ~~ 10 full 90 minute games (standard threshold used in football analytics)
df_merged = df_merged[df_merged['minutes'] >= 900]
print(df_merged.shape)

# Dropping redundant columns - we already have age and minutes (90s is minutes/90) so we can safely exclude born and 90s
df_merged = df_merged.drop(columns=['born', '90s'])
print("Final Shape:", df_merged.shape)

# Saving the merged and cleaned data to a new csv file for further processing
df_merged.to_csv("../data/processed/merged.csv", index=False)
print("Saved!")


In [ ]:
print(df_merged['pos'].isna().sum()) # Checking for NaN values
print(df_merged['pos'].unique()) # Checking for all the different unique positions under the pos column

0
<StringArray>
['DF', 'FW', 'GK', 'MF,DF', 'MF', 'DF,MF', 'MF,FW', 'FW,MF', 'DF,FW']
Length: 9, dtype: str


In [ ]:
# Creating a separata dataframe to store the attackers (people who contribute to goals) only
df_attackers = df_merged[df_merged.str.contains('FW') | df_merged.str.contains('MF')]